In [ ]:
import os
import sys

import torch
from ultralytics import YOLO

# Add project root to system path (for relative imports to work)
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src import config

torch.cuda.empty_cache()

In [ ]:
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

weights_path = (
    f"{config.WORKSPACE_ROOT}/data/runs/yolo/train/motor_yolov8x_1280_long/weights/best.pt"
)

model = YOLO(weights_path)

image_glob = "/data/horse/ws/kein254g-team_project/test/**/*.jpg"
save_dir_project = "/data/horse/ws/kein254g-team_project/yolo_batch"
save_dir_name = "exp"
device_str = "cuda:0" if torch.cuda.is_available() else "cpu"


try:
    results = model.predict(
        source=image_glob,
        imgsz=960,
        conf=0.6,
        device=device_str,
        half=True,
        batch=4,
        workers=0,
        stream=False,
        save=True,
        save_txt=True,
        save_conf=True,
        project=save_dir_project,
        name=save_dir_name,
    )
except RuntimeError as e:
    msg = str(e)
    print("RuntimeError:", msg)
    if "CUDA out of memory" in msg or "CUDNN_STATUS_ALLOC_FAILED" in msg:
        print("\nOOM detected. Retrying with smaller settings...")
        torch.cuda.empty_cache()
        # predict_args.update(dict(imgsz=768, batch=1, half=True))
        # results = model.predict(**predict_args)
    else:
        raise